# Model 2 — Keras Squat Classifier
Auto-labels frames from downloaded videos using Model 1 angle logic, then trains a neural network.
Exports `.keras`, scaler JSON, and `.tflite`.

In [1]:
%pip install mediapipe opencv-python numpy tensorflow scikit-learn kagglehub

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: C:\Users\gusta\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


## Step 1 — Setup

In [2]:
import cv2
import mediapipe as mp
import numpy as np
import os
import json
import shutil
import urllib.request
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision

MODEL_PATH = os.path.join(os.getcwd(), 'pose_landmarker.task')

if not os.path.exists(MODEL_PATH):
    print('Downloading pose landmarker model...')
    urllib.request.urlretrieve(
        'https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_lite/float16/latest/pose_landmarker_lite.task',
        MODEL_PATH
    )
    print('Done.')
else:
    print(f'Model found: {MODEL_PATH}')

LEFT_SHOULDER = 11
LEFT_HIP      = 23
LEFT_KNEE     = 25
LEFT_ANKLE    = 27
RIGHT_HIP     = 24
RIGHT_KNEE    = 26
RIGHT_ANKLE   = 28

SQUAT_CONFIG = {
    'squat_knee_threshold':    120,
    'standing_knee_threshold': 150,
    'hip_angle_threshold':     130,
    'visibility_threshold':    0.6,
}

def calculate_angle(a, b, c):
    a, b, c = np.array(a), np.array(b), np.array(c)
    ba, bc = a - b, c - b
    cosine = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-6)
    return np.degrees(np.arccos(np.clip(cosine, -1.0, 1.0)))

def auto_label(landmarks):
    def get(idx):
        p = landmarks[idx]
        return [p.x, p.y], p.visibility
    sh_l, v1 = get(LEFT_SHOULDER)
    hi_l, v2 = get(LEFT_HIP)
    kn_l, v3 = get(LEFT_KNEE)
    an_l, v4 = get(LEFT_ANKLE)
    hi_r, v5 = get(RIGHT_HIP)
    kn_r, v6 = get(RIGHT_KNEE)
    an_r, v7 = get(RIGHT_ANKLE)
    if min(v1,v2,v3,v4,v5,v6,v7) < SQUAT_CONFIG['visibility_threshold']:
        return -1, None
    knee_l   = calculate_angle(hi_l, kn_l, an_l)
    knee_r   = calculate_angle(hi_r, kn_r, an_r)
    hip_l    = calculate_angle(sh_l, hi_l, kn_l)
    avg_knee = (knee_l + knee_r) / 2
    flat = [v for lm in landmarks for v in (lm.x, lm.y, lm.visibility)]
    if avg_knee < SQUAT_CONFIG['squat_knee_threshold'] and hip_l < SQUAT_CONFIG['hip_angle_threshold']:
        return 1, flat
    elif avg_knee > SQUAT_CONFIG['standing_knee_threshold']:
        return 0, flat
    return -1, None

print('Setup complete.')

Model found: c:\Users\gusta\Desktop\SkoleCoding\ML_NODE_EXAM\PulseML.js\ML\Squats\pose_landmarker.task
Setup complete.


## Step 2 — Download training videos from Kaggle

In [3]:
import kagglehub

VIDEOS_DIR = os.path.join(os.getcwd(), 'trainingData', 'videos')
os.makedirs(VIDEOS_DIR, exist_ok=True)

dataset_path = kagglehub.dataset_download('hasyimabdillah/workoutfitness-video')
print('Downloaded to:', dataset_path)

squat_folder = os.path.join(dataset_path, 'squat')
if not os.path.exists(squat_folder):
    for root, dirs, files in os.walk(dataset_path):
        if os.path.basename(root).lower() == 'squat':
            squat_folder = root
            break

print('Squat folder:', squat_folder)

for fname in os.listdir(squat_folder):
    if fname.endswith(('.mp4', '.mov', '.avi')):
        src = os.path.join(squat_folder, fname)
        dst = os.path.join(VIDEOS_DIR, fname)
        if not os.path.exists(dst):
            shutil.copy2(src, dst)
            print(f'Copied: {fname}')
        else:
            print(f'Already exists: {fname}')

print(f'\nTotal videos ready: {len(os.listdir(VIDEOS_DIR))}')

C:\Users\gusta\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Downloaded to: C:\Users\gusta\.cache\kagglehub\datasets\hasyimabdillah\workoutfitness-video\versions\5
Squat folder: C:\Users\gusta\.cache\kagglehub\datasets\hasyimabdillah\workoutfitness-video\versions\5\squat
Already exists: squat_10.mp4
Already exists: squat_11.mp4
Already exists: squat_12.mp4
Already exists: squat_13.mp4
Already exists: squat_14.mp4
Already exists: squat_15.mp4
Already exists: squat_16.mp4
Already exists: squat_17.mp4
Already exists: squat_18.mp4
Already exists: squat_19.mp4
Already exists: squat_20.mp4
Already exists: squat_21.mp4
Already exists: squat_22.mp4
Already exists: squat_23.mp4
Already exists: squat_24.mp4
Already exists: squat_25.mp4
Already exists: squat_26.mp4
Already exists: squat_27.mp4
Already exists: squat_28.mp4
Already exists: squat_29.mp4
Already exists: squat_7.mp4
Already exists: squat_8.mp4
Already exists: squat_9.mp4

Total videos ready: 23


## Step 3 — Extract and auto-label frames

In [4]:
VIDEO_PATHS = [
    os.path.join(VIDEOS_DIR, f)
    for f in os.listdir(VIDEOS_DIR)
    if f.endswith(('.mp4', '.mov', '.avi'))
]
print(f'Found {len(VIDEO_PATHS)} videos')

def extract_and_label(video_path):
    X, y = [], []
    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    base_options = mp_python.BaseOptions(model_asset_path=MODEL_PATH)
    options = mp_vision.PoseLandmarkerOptions(
        base_options=base_options,
        running_mode=mp_vision.RunningMode.VIDEO,
    )
    timestamp_ms = 0
    processed = 0
    with mp_vision.PoseLandmarker.create_from_options(options) as landmarker:
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
            timestamp_ms += int(1000 / 30)
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
            result = landmarker.detect_for_video(mp_image, timestamp_ms)
            if result.pose_landmarks:
                label, flat = auto_label(result.pose_landmarks[0])
                if label != -1:
                    X.append(flat)
                    y.append(label)
            processed += 1
            if processed % 100 == 0:
                print(f'  {processed}/{total} frames')
    cap.release()
    return X, y

X_all, y_all = [], []
for path in VIDEO_PATHS:
    print(f'Processing {os.path.basename(path)}...')
    X_v, y_v = extract_and_label(path)
    X_all.extend(X_v)
    y_all.extend(y_v)

X = np.array(X_all, dtype=np.float32)
y = np.array(y_all, dtype=np.int32)
print(f'\nDataset: {len(X)} frames — standing: {np.sum(y==0)}, squat: {np.sum(y==1)}')

Found 23 videos
Processing squat_10.mp4...
  100/175 frames
Processing squat_11.mp4...
  100/144 frames
Processing squat_12.mp4...
  100/113 frames
Processing squat_13.mp4...
  100/130 frames
Processing squat_14.mp4...
  100/101 frames
Processing squat_15.mp4...
  100/280 frames
  200/280 frames
Processing squat_16.mp4...
Processing squat_17.mp4...
  100/110 frames
Processing squat_18.mp4...
Processing squat_19.mp4...
  100/143 frames
Processing squat_20.mp4...
  100/233 frames
  200/233 frames
Processing squat_21.mp4...
  100/348 frames
  200/348 frames
  300/348 frames
Processing squat_22.mp4...
  100/380 frames
  200/380 frames
  300/380 frames
Processing squat_23.mp4...
  100/432 frames
  200/432 frames
  300/432 frames
  400/432 frames
Processing squat_24.mp4...
  100/198 frames
Processing squat_25.mp4...
  100/338 frames
  200/338 frames
  300/338 frames
Processing squat_26.mp4...
  100/368 frames
  200/368 frames
  300/368 frames
Processing squat_27.mp4...
  100/647 frames
  200

## Step 4 — Train

In [5]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from tensorflow import keras

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

model = keras.Sequential([
    keras.layers.Input(shape=(99,)),
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(64, activation='relu'),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(1, activation='sigmoid'),
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

history = model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=30, batch_size=32)

loss, acc = model.evaluate(X_test, y_test)
print(f'Test accuracy: {acc:.3f}')

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │        12,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 21,121 (82.50 KB)

 Trainable params: 21,121 (82.50 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/30
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.8238 - loss: 0.4594 - val_accuracy: 0.9072 - val_loss: 0.2542
Epoch 2/30
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9557 - loss: 0.1801 - val_accuracy: 0.9367 - val_loss: 0.1497
Epoch 3/30
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9726 - loss: 0.0967 - val_accuracy: 0.9494 - val_loss: 0.1041
Epoch 4/30
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9863 - loss: 0.0693 - val_accuracy: 0.9536 - val_loss: 0.1234
Epoch 5/30
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9873 - loss: 0.0530 - val_accuracy: 0.9789 - val_loss: 0.0585
Epoch 6/30
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9863 - loss: 0.0482 - val_accuracy: 0.9831 - val_loss: 0.0482
Epoch 7/30
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9884 - loss: 0.0380 - val_accuracy: 0.9536 - val_loss: 0.0794
Epoch 8/30
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9852 - loss: 0.0442 - val_accuracy: 0.9831 - val_loss:

## Step 5 — Export

In [6]:
model.save('squat_model2.keras')
print('Exported: squat_model2.keras')

scaler_params = {'mean': scaler.mean_.tolist(), 'scale': scaler.scale_.tolist()}
with open('squat_model2_scaler.json', 'w') as f:
    json.dump(scaler_params, f)
print('Exported: squat_model2_scaler.json')

converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()
with open('squat_model2.tflite', 'wb') as f:
    f.write(tflite_model)
print('Exported: squat_model2.tflite')

Exported: squat_model2.keras
Exported: squat_model2_scaler.json
INFO:tensorflow:Assets written to: C:\Users\gusta\AppData\Local\Temp\tmpooulkyvw\assets


INFO:tensorflow:Assets written to: C:\Users\gusta\AppData\Local\Temp\tmpooulkyvw\assets


Saved artifact at 'C:\Users\gusta\AppData\Local\Temp\tmpooulkyvw'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 99), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  1581692622160: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1581692623504: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1581692621968: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1581692623120: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1581692623312: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1581692623888: TensorSpec(shape=(), dtype=tf.resource, name=None)
Exported: squat_model2.tflite


## Step 6 — Live inference

In [7]:
THRESHOLD = 0.5

base_options = mp_python.BaseOptions(model_asset_path=MODEL_PATH)
options = mp_vision.PoseLandmarkerOptions(
    base_options=base_options,
    running_mode=mp_vision.RunningMode.VIDEO,
)

cap = cv2.VideoCapture(0)
squat_count = 0
in_squat = False
timestamp_ms = 0

with mp_vision.PoseLandmarker.create_from_options(options) as landmarker:
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        timestamp_ms += int(1000 / 30)
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
        result = landmarker.detect_for_video(mp_image, timestamp_ms)
        if result.pose_landmarks:
            flat = [v for lm in result.pose_landmarks[0] for v in (lm.x, lm.y, lm.visibility)]
            prob = model.predict(scaler.transform([flat]), verbose=0)[0][0]
            is_squat = prob > THRESHOLD
            if is_squat and not in_squat:
                in_squat = True
            elif not is_squat and in_squat:
                squat_count += 1
                in_squat = False
            label = f'SQUAT {prob:.2f}' if is_squat else f'STANDING {1-prob:.2f}'
            color = (0, 255, 0) if is_squat else (0, 0, 255)
            cv2.putText(frame, f'{label}  Reps: {squat_count}', (20, 50),
                        cv2.FONT_HERSHEY_SIMPLEX, 1.1, color, 2)
        cv2.imshow('Squat Detector - Model 2', frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()